# 🧠 Non-Maximum Suppression (NMS)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Non-Maximum Suppression (NMS)**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายว่าทำไม NMS จึงมีความสำคัญอย่างยิ่งสำหรับการทำ post-processing ในแบบจำลองการตรวจจับวัตถุ (เช่น YOLO)
2. สร้างอัลกอริทึม NMS มาตรฐานจากศูนย์ (from scratch) โดยใช้ NumPy
3. กำหนดกรณีกล่องขอบเขต (bounding box) ที่เป็นรูปธรรมซึ่งมีการคาดการณ์ที่ทับซ้อนกันและไม่ทับซ้อนกัน
4. ใช้ NMS เพื่อกรองการตรวจจับที่ซ้ำซ้อนออกไป
5. พลอตกล่องตัวเลือกดั้งเดิมและการเน้นกล่องที่เก็บไว้เทียบกับกล่องที่ถูกคัดออกโดยใช้ Matplotlib
6. แนะนำ **Soft-NMS** ซึ่งเป็นวิธีแก้ปัญหาขั้นสูงในการจัดการวัตถุที่ทับซ้อนกันแต่เป็นคนละวัตถุ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Scratch NMS Implementation in NumPy

นี่คืออัลกอริทึม hard NMS มาตรฐานที่ถูกสร้างขึ้นโดยใช้การดำเนินการแบบเวกเตอร์ (vectorised operations) ใน NumPy ซึ่งจะวนลูปผ่านกล่องต่าง ๆ ที่เรียงลำดับตามคะแนนความมั่นใจ (confidence score) จากนั้นเลือกกล่องที่มีคะแนนสูงสุดอย่างตะกละ (greedily) และคัดกล่องอื่น ๆ ที่ทับซ้อนกับกล่องดังกล่าวเกินเกณฑ์ที่กำหนด (threshold) ออกไป

In [ ]:
def nms(boxes, scores, iou_threshold):
    if len(boxes) == 0:
        return []
        
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        
        if order.size == 1:
            break
            
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        intersection = w * h
        
        union = areas[i] + areas[order[1:]] - intersection
        iou = intersection / union
        
        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]
        
    return keep

## 2. Defining Bounding Box Predictions

เราได้ตั้งค่ากล่องขอบเขตที่เป็นตัวเลือก 5 กล่องพร้อมคะแนนความมั่นใจที่สอดคล้องกัน โดยบางกล่องแสดงถึงการคาดการณ์ที่ทับซ้อนกันอย่างมากสำหรับวัตถุชิ้นเดียวกัน ในขณะที่กล่องอื่น ๆ แสดงถึงวัตถุที่แยกจากกันชัดเจน

In [ ]:
boxes = np.array([
    [100.0, 100.0, 200.0, 200.0],  # Box 0 (score: 0.90)
    [105.0, 105.0, 205.0, 205.0],  # Box 1 (score: 0.80) - highly overlaps Box 0
    [100.0, 100.0, 150.0, 150.0],  # Box 2 (score: 0.40) - low overlap Box 0
    [300.0, 300.0, 400.0, 400.0],  # Box 3 (score: 0.95)
    [310.0, 310.0, 410.0, 410.0]   # Box 4 (score: 0.85) - highly overlaps Box 3
])
scores = np.array([0.90, 0.80, 0.40, 0.95, 0.85])
iou_threshold = 0.5

keep_indices = nms(boxes, scores, iou_threshold)
print("Kept indices:", [int(x) for x in keep_indices])

## 3. Visualizing NMS Results

มาแสดงภาพผลลัพธ์ของกล่องกัน โดยกล่องสีเขียว/เส้นทึบแสดงถึงการตรวจจับที่เก็บไว้ (kept detections) และกล่องสีแดง/เส้นประแสดงถึงการตรวจจับที่ถูกคัดออก (suppressed detections)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(50, 450)
ax.set_ylim(450, 50)  # Inverted y-axis to match typical image coordinate system

for idx in range(len(boxes)):
    box = boxes[idx]
    w = box[2] - box[0]
    h = box[3] - box[1]
    score = scores[idx]
    
    if idx in keep_indices:
        rect = patches.Rectangle((box[0], box[1]), w, h, linewidth=3, edgecolor='green', facecolor='none')
        ax.add_patch(rect)
        ax.text(box[0] + 5, box[1] + 20, f"Box {idx} ({score:.2f}) [KEPT]", color='green', fontweight='bold', fontsize=10)
    else:
        rect = patches.Rectangle((box[0], box[1]), w, h, linewidth=2, edgecolor='red', linestyle='--', facecolor='none', alpha=0.6)
        ax.add_patch(rect)
        ax.text(box[0] + 5, box[1] + 20, f"Box {idx} ({score:.2f}) [SUPPRESSED]", color='red', fontsize=10)

ax.set_title("Non-Maximum Suppression (NMS) Results (IoU threshold = 0.5)", fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. Soft-NMS: Handling Occlusions and Side-by-Side Objects

Hard NMS จะตัดกล่องขอบเขตที่ทับซ้อนกันออกทั้งหมดหากค่า IoU สูงเกินเกณฑ์ที่กำหนด ในสถานการณ์ที่วัตถุหนาแน่นและอยู่เบียดกัน (เช่น ส่วนประกอบที่อยู่เคียงข้างกัน หรือคนเดินถนนที่เดินใกล้กันมาก) วิธีนี้จะทำให้เกิดการพลาดการตรวจจับ (False Negatives)

**Soft-NMS** จะลดทอนคะแนนการตรวจจับของกล่องที่ทับซ้อนกันลงตามฟังก์ชันต่อเนื่องของพื้นที่ทับซ้อน แทนที่จะปรับลดให้เป็นศูนย์ในทันที กฎการอัปเดตคะแนนคือ:
$$s_i \leftarrow s_i \cdot \exp\left( -\frac{\text{IoU}(M, b_i)^2}{\sigma} \right)$$

หากคะแนนที่ลดทอนลงแล้วยังคงสูงกว่าเกณฑ์คะแนนขั้นที่สอง วัตถุนั้นก็จะถูกเก็บรักษาไว้! วิธีนี้ช่วยให้ตัวตรวจจับวัตถุสามารถค้นพบวัตถุได้ครบถ้วนยิ่งขึ้น (recall ดีขึ้น) ในสภาพแวดล้อมที่หนาแน่น